# Meridian Commerce Customer Support Chat

FAQ: Cache; Chunking -> FAISS embedding -> HyDE Optimization -> biencoding -> crossencoding / reranking -> Augmentation -> LLM

In [4]:
import os
import pickle
from pathlib import Path

import numpy as np
import faiss
import anthropic

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder


In [5]:
VECTOR_DIR = Path("./faiss_policy_index")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

INDEX_FILE = VECTOR_DIR / "policy.index"
CHUNKS_FILE = VECTOR_DIR / "policy_chunks.pkl"


In [6]:
load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=api_key)


In [27]:
# Chunking policy files: Recursive chunking
import re
from pathlib import Path


POLICY_DIR = Path("./resources/CustomerSupportData/PolicyData")

policy_chunks = []

overlap = ""


for file_path in POLICY_DIR.glob("*.txt"):

    text = file_path.read_text(
        encoding="utf-8"
    ).strip()

    # Normalize line endings
    text = text.replace("\r\n", "\n")

    # Remove empty lines
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    # First line is the document title
    document_title = lines[0]

    # Remaining text contains the numbered policies
    policy_text = "\n".join(lines[1:])

    # Split recursively by top-level decimal numbering
    policies = re.split(
        r"(?m)(?=^\d+\.\s+)",
        policy_text
    )

    policies = [
        policy.strip()
        for policy in policies
        if policy.strip()
    ]

    # Reset overlap for every document
    overlap = ""

    for index, policy in enumerate(policies):

        # Extract policy number
        match = re.match(
            r"^(\d+)\.",
            policy
        )

        if match:
            policy_number = match.group(1)
        else:
            policy_number = str(index + 1)

        # Current policy always stays complete.
        # Previous policy is added as overlap/context.
        chunk_text = (
            f"Document: {document_title}\n\n"
            f"{overlap}"
            f"Current Policy {policy_number}:\n"
            f"{policy}"
        )

        policy_chunks.append(
            {
                "id": f"{file_path.stem}_policy_{policy_number}",
                "text": chunk_text,
                "source_file": file_path.name,
                "document_title": document_title,
                "policy_number": policy_number
            }
        )

        # Saving the complete current policy as overlap for the next one
        overlap = (
            f"Previous Policy Context:\n"
            f"{policy}\n\n"
        )


print(f"Created {len(all_chunks)} policy chunks\n")

for chunk in policy_chunks[:3]:
    # Displaying top 3 chunks
    print("=" * 100)
    print("ID:", chunk["id"])
    print("SOURCE:", chunk["source_file"])
    print("POLICY:", chunk["policy_number"])
    print()
    print(chunk["text"])

Created 32 policy chunks

ID: policy_loyalty_program_policy_1
SOURCE: policy_loyalty_program.txt
POLICY: 1

Document: LOYALTY PROGRAM TERMS — Meridian Rewards

Current Policy 1:
1. Enrollment
Meridian Rewards is free to join and every registered customer is
automatically enrolled from their first completed order. No separate
sign-up is required.
ID: policy_loyalty_program_policy_2
SOURCE: policy_loyalty_program.txt
POLICY: 2

Document: LOYALTY PROGRAM TERMS — Meridian Rewards

Previous Policy Context:
1. Enrollment
Meridian Rewards is free to join and every registered customer is
automatically enrolled from their first completed order. No separate
sign-up is required.

Current Policy 2:
2. Earning Points
Customers earn 1 Meridian Point for every ₹100 spent on eligible orders.
Points are credited once an order is marked "Delivered" and are reversed if
the corresponding item is later returned or refunded.
ID: policy_loyalty_program_policy_3
SOURCE: policy_loyalty_program.txt
POLICY: 3

D

In [24]:
# Loading FAQs
def load_faq_cache(file_path):

    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"FAQ file not found: {file_path}"
        )

    text = file_path.read_text(
        encoding="utf-8"
    ).strip()

    cache = {}

    # Normalize line endings
    text = text.replace("\r\n", "\n")

    # Split whenever a new FAQ question begins.
    # (?m) allows ^Q: to match at the beginning of every line.
    faq_blocks = re.split(
        r"(?m)^\s*Q:\s*",
        text
    )

    for block in faq_blocks:

        block = block.strip()

        if not block:
            continue

        # Split the block into question and answer
        parts = re.split(
            r"(?m)^\s*A:\s*",
            block,
            maxsplit=1
        )

        # Skip malformed blocks
        if len(parts) != 2:
            continue

        question = parts[0].strip()
        answer = parts[1].strip()

        # Convert multi-line question/answer into clean single strings
        question = " ".join(
            line.strip()
            for line in question.splitlines()
            if line.strip()
        )

        answer = " ".join(
            line.strip()
            for line in answer.splitlines()
            if line.strip()
        )

        if question and answer:
            cache[question] = answer

    if not cache:
        raise ValueError(
            "No FAQs could be parsed. "
            "Expected format:\n"
            "Q: question\n"
            "A: answer"
        )

    return cache


# Load FAQ cache
faq_cache = load_faq_cache(
    "./resources/CustomerSupportData/policy_payment_refunds_faq.txt"
)

print(f"Loaded {len(faq_cache)} FAQ entries\n")

i = 1

for question, answer in faq_cache.items():
    print(f"{i}. Q: {question}")
    print(f"   A: {answer}\n")
    i += 1

Loaded 9 FAQ entries

1. Q: What payment methods does Meridian Commerce accept?
   A: We accept credit/debit cards, UPI, net banking, popular mobile wallets, and Cash on Delivery (COD) on eligible orders below ₹15,000.

2. Q: I was charged twice for one order. What do I do?
   A: This is usually a temporary hold from your bank that resolves within 3-5 business days without any action needed. If a duplicate charge is still visible on your statement after 5 business days, contact Support with your order ID and both transaction references, and we will refund the duplicate charge within 3 business days of verification.

3. Q: My payment failed but the amount was deducted from my account.
   A: Failed-payment deductions are automatically reversed by your bank or payment provider within 5-7 business days. If the amount is not reversed after 7 business days, contact Support with your order ID and transaction ID for manual investigation.

4. Q: How long does a refund take once approved?
   A: 

In [25]:
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2636.44it/s]


In [28]:
# Creating FAISS index
policy_texts = [item["text"] for item in policy_chunks]

embeddings = bi_encoder.encode(
    policy_texts,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

# Normalized vectors + IndexFlatIP = cosine similarity search.
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

faiss.write_index(index, str(INDEX_FILE))

with open(CHUNKS_FILE, "wb") as f:
    pickle.dump(policy_chunks, f)

print("FAISS vectors saved:", index.ntotal)
print("Index :", INDEX_FILE)
print("Chunks:", CHUNKS_FILE)


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.56s/it]

FAISS vectors saved: 32
Index : faiss_policy_index\policy.index
Chunks: faiss_policy_index\policy_chunks.pkl


In [29]:
# HyDE Prompt optimization
def generate_hyde_document(query):
    prompt = f"""
You are optimizing retrieval for a Meridian Commerce customer-support
knowledge base.

Customer question:
{query}

Write a short hypothetical policy passage that would ideally contain the
information needed to answer this question.

Rules:
- Write in neutral policy/document style.
- Include relevant concepts, conditions, and terminology.
- Do NOT pretend invented details are actual Meridian policy.
- Do NOT answer the customer conversationally.
- Keep it under 180 words.
- Return only the hypothetical passage.
- If the customer question does not refer to policies then return the query as it is in the response without modification
"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=250,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.content[0].text.strip()


In [30]:
# biencoder and crossencoder
def retrieve_and_rerank(query, top_k=10, top_n=4):
    # 1. HyDE creates a retrieval-oriented hypothetical document.
    hyde_document = generate_hyde_document(query)

    # 2. Bi-encoder embeds HyDE output.
    query_embedding = bi_encoder.encode(
        [hyde_document],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    # 3. FAISS initial cosine similarity retrieval.
    scores, indices = index.search(
        query_embedding,
        min(top_k, index.ntotal)
    )

    candidates = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        item = policy_chunks[int(idx)].copy()
        item["bi_encoder_score"] = float(score)
        candidates.append(item)

    # 4. Cross-encoder reranking uses the original customer query.
    pairs = [(query, item["text"]) for item in candidates]

    cross_scores = cross_encoder.predict(pairs)

    for item, score in zip(candidates, cross_scores):
        item["cross_encoder_score"] = float(score)

    candidates.sort(
        key=lambda x: x["cross_encoder_score"],
        reverse=True
    )

    return hyde_document, candidates[:top_n]


In [34]:
# Augment And Generate
def generate_rag_answer(query, retrieved_chunks):
    context = "\n\n".join(
        f"POLICY CONTENT:\n{item['text']}"
        for item in retrieved_chunks
    )

    prompt = f"""
You are the Meridian Commerce customer support assistant.

Answer the customer using ONLY the retrieved policy context below.

Rules:
1. Do not invent policies, dates, monetary amounts, thresholds, or exceptions.
2. Do not use outside knowledge.
3. If the retrieved context is insufficient, clearly say that the provided
   policy documents do not contain enough information.
4. Do not mention FAISS, embeddings, HyDE, reranking, or internal systems.
5. Be concise, clear, and customer-friendly.

RETRIEVED POLICY CONTEXT:
{context}

CUSTOMER QUESTION:
{query}
"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=450,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.content[0].text.strip()


In [35]:
# Test
query = "What happens if my delivery is delayed?"

hyde_document, results = retrieve_and_rerank(
    query,
    top_k=10,
    top_n=4
)

print("CUSTOMER QUERY:\n", query)
print("\nHYDE DOCUMENT:\n", hyde_document)

print("\nRETRIEVED RESULTS:")
for i, item in enumerate(results, start=1):
    print(
        f"Bi-Encoder Cosine: {item['bi_encoder_score']:.4f} | "
        f"Cross-Encoder: {item['cross_encoder_score']:.4f}"
    )

answer = generate_rag_answer(query, results)

print("\nFINAL RAG ANSWER:\n", answer)


CUSTOMER QUERY:
 What happens if my delivery is delayed?

HYDE DOCUMENT:
 **Delivery Delay Policy**

In the event of a delivery delay, Meridian Commerce will notify customers via email or SMS using the contact information provided at checkout. Notifications are typically sent when a shipment is expected to arrive later than the original estimated delivery date.

Customers may be eligible for the following remedies depending on the circumstances:

- **Carrier delays**: If the delay is caused by the shipping carrier, customers should contact the carrier directly using the tracking number provided. Meridian Commerce will assist in filing claims when applicable.

- **Warehouse or fulfillment delays**: If the delay originates from our facilities, customers may be offered expedited shipping at no additional charge or a partial refund at our discretion.

- **Extended delays**: For delays exceeding the timeframe specified in our shipping terms, customers may request order cancellation and full